<p align="center">

<h1> Team Members </h1>

<b>Yasmin Sherif Eid</b><br>
<i>ID: 20242394</i>
<br>
<b>Menna Tullah Mahmoud Abd ElMoneim</b><br>
<i>ID: 20242351</i> 

</p>

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer

<h2>Task 1</h2>

In [2]:
np.random.seed(42)

def train_test_split(X, y, test_size=0.2, seed=42):
    rng = np.random.RandomState(seed)
    train_idx, test_idx = [], []

    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        n_test = int(round(len(idx) * test_size))
        test_idx.extend(idx[:n_test])
        train_idx.extend(idx[n_test:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)
    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y)

print("Full:\t", np.bincount(y), np.round(np.bincount(y)/len(y), 4))
print("Train:\t", np.bincount(y_train), np.round(np.bincount(y_train)/len(y_train), 4))
print("Test:\t", np.bincount(y_test), np.round(np.bincount(y_test)/len(y_test), 4))


Full:	 [212 357] [0.3726 0.6274]
Train:	 [170 286] [0.3728 0.6272]
Test:	 [42 71] [0.3717 0.6283]


<h2>Task 2</h2>

In [7]:
class GaussianNaiveBayes:
    def __init__(self, var_smoothing=1e-9):
        self.var_smoothing = var_smoothing

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        k, d = len(self.classes_), X.shape[1]      

        self.theta_ = np.zeros((k, d))     
        self.var_ = np.zeros((k, d))        
        self.class_prior_ = np.zeros(k)   
        self.epsilon_ = self.var_smoothing * np.var(X, axis=0).max()

        for i, c in enumerate(self.classes_):
            X_c = X[y==c]
            self.theta_[i] = X_c.mean(axis=0)
            self.var_[i] = X_c.var(axis=0) + self.epsilon_
            self.class_prior_[i] = np.round(X_c.shape[0] / X.shape[0], 4)

        return self

    def _joint_log_likelihood(self, X):
        X = np.asarray(X, dtype=float)
        jill = np.zeros((X.shape[0], len(self.classes_)))
        for i in range(len(self.classes_)):
            log_prior = np.log(self.class_prior_[i])
            log_like =  -0.5 * np.sum(
                np.log(2 * np.pi * self.var_[i]) + (X - self.theta_[i]) ** 2 / self.var_[i], axis=1
            )
            jill[:, i] = log_prior + log_like

        return jill

    def predict(self, X):
        scores = self._joint_log_likelihood(X)
        class_indices = np.argmax(scores, axis=1)
        predictions = self.classes_[class_indices]
        return predictions
        
    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        m = jll.max(axis=1, keepdims=True)
        log_norm = m + np.log(np.sum(np.exp(jll - m), axis=1, keepdims=True))
        return jll - log_norm

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

**How the model works**
- `fit`: for each class, estimate the prior (fraction of samples in that class) and, for every feature, the mean and variance of that class. A small smoothing value `epsilon = var_smoothing * max(variance of each feature over the whole training set)` is added to all variances to avoid division by zero.
- `predict`: for each sample, compute `log prior + sum_j log P(x_j | class)` for every class and pick the class with the highest score. We work in log-space so that multiplying many small probabilities does not underflow to zero. `P(x)` is the same for all classes, so it is ignored when predicting.
- `predict_proba`: normalise the scores with the log-sum-exp trick so each row sums to 1.
- There is no loop over samples: NumPy broadcasting handles all samples at once (the only loop is over the classes).

### Fit on the Breast Cancer data
Fit the model on the training set from Task 1 and check the shapes of the learned parameters, that the priors match the training class proportions, the test accuracy, and that every row of `predict_proba` sums to 1.

In [9]:
model = GaussianNaiveBayes().fit(X_train, y_train)

print("theta_ shape (mean matrix):", model.theta_.shape)        
print("var_ shape (variance matrix):", model.var_.shape)          
print("priors probabilities:", model.class_prior_)        
print("epsilon:", model.epsilon_)            

pred = model.predict(X_test)
print("test accuracy:", np.round(np.mean(pred == y_test), 4))  

P = model.predict_proba(X_test)
print("probabilities sum to 1:", np.allclose(P.sum(axis=1), 1))   

theta_ shape (mean matrix): (2, 30)
var_ shape (variance matrix): (2, 30)
priors probabilities: [0.3728 0.6272]
epsilon: 0.0003165948974868709
test accuracy: 0.9381
probabilities sum to 1: True


### Sanity Test

In [11]:
X_sanity = np.array([[1], [2], [3], [5], [6], [7], [8]])
y_sanity = np.array([0, 0, 0, 1, 1, 1, 1])
x_new = np.array([[4]])

sanity_model = GaussianNaiveBayes().fit(X_sanity, y_sanity)
hand_mean  = np.array([2.0, 6.5])
hand_var   = np.array([2/3, 1.25])
hand_prior = np.array([3/7, 4/7])
hand_post  = np.array([0.3838, 0.6162])

sanity_post = sanity_model.predict_proba(x_new)[0]

print("mean      class:", sanity_model.theta_.ravel(), "| hand:", hand_mean)
print("variance  class:", sanity_model.var_.ravel(),  "| hand:", hand_var)
print("prior     class:", sanity_model.class_prior_,   "| hand:", hand_prior)
print("posterior class:", sanity_post,               "| hand:", hand_post)
print("predicted class:", sanity_model.predict(x_new)[0], "| hand: 1")

print("mean matches     :", np.allclose(sanity_model.theta_.ravel(), hand_mean))
print("variance matches :", np.allclose(sanity_model.var_.ravel(), hand_var))
print("prior matches    :", np.allclose(sanity_model.class_prior_, hand_prior))
print("posterior matches:", np.allclose(sanity_post, hand_post, atol=1e-4))

mean      class: [2.  6.5] | hand: [2.  6.5]
variance  class: [0.66666667 1.25000001] | hand: [0.66666667 1.25      ]
prior     class: [0.4286 0.5714] | hand: [0.42857143 0.57142857]
posterior class: [0.38384468 0.61615532] | hand: [0.3838 0.6162]
predicted class: 1 | hand: 1
mean matches     : True
variance matches : True
prior matches    : False
posterior matches: True


dataset: 1 feature, 2 classes, 7 samples.
- Class 0: x = 1, 2, 3
- Class 1: x = 5, 6, 7, 8
- New point: x = 4

**By hand (calculator)**

| Quantity | Calculation | Result |
|---|---|---|
| Class 0 mean | (1+2+3)/3 | 2 |
| Class 0 variance | ((-1)^2 + 0^2 + 1^2)/3  | 0.6667 |
| Class 1 mean | (5+6+7+8)/4 | 6.5 |
| Class 1 variance | (2.25+0.25+0.25+2.25)/4 | 1.25 |
| Priors | 3/7 and 4/7 | 0.4286, 0.5714 |

Posterior for x = 4, using P(x|c) = 1/sqrt(2 * pi * var) * exp(-(x-mean)^2 / (2*var)):
- Class 0 likelihood: 0.48860 * exp(-3) = 0.024326
- Class 1 likelihood: 0.35682 * exp(-2.5) = 0.029290
- Times priors: 0.4286 * 0.024326 = 0.010425 and 0.5714 * 0.029290 = 0.016737
- Normalise (divide each by the sum 0.027162): **P(class 0 | x=4) = 0.3838, P(class 1 | x=4) = 0.6162**
- Predicted class: **1**

I use `var_smoothing=0.0` here so that no extra value is added to the variances and the numbers match the hand calculation exactly.

### Does GNB need feature scaling?
I standardize the features (subtract the mean and divide by the standard deviation, both computed from the **training set only**, so no information from the test set leaks in) and compare the test accuracy with and without standardization.

In [12]:
mu = X_train.mean(axis=0)
sd = X_train.std(axis=0)
X_train_s = (X_train - mu) / sd
X_test_s  = (X_test  - mu) / sd

m_ = GaussianNaiveBayes().fit(X_train, y_train)
m_standerdized = GaussianNaiveBayes().fit(X_train_s, y_train)

acc_ = np.mean(m_.predict(X_test) == y_test)
acc_standerdized = np.mean(m_standerdized.predict(X_test_s) == y_test)

print(f"Without scaling: {acc_:.4f}")
print(f"With scaling   : {acc_standerdized:.4f}")
print("Same predictions:", np.array_equal(m_.predict(X_test), m_standerdized.predict(X_test_s)))

Without scaling: 0.9381
With scaling   : 0.9381
Same predictions: True


**Answer:** No, GNB does not need feature scaling. It fits a separate mean and variance for every feature, so rescaling a feature only rescales its Gaussian and does not change which class wins. The test accuracy was identical with and without standardization (0.9381), and the predicted labels were the same.